# NER Token Classification — Baseline Encoder → BIO

Fine-tune a pre-trained encoder (BERT / RoBERTa / DeBERTa) for Named Entity Recognition on the **FIRE** financial dataset using **BIO tagging** with **first-subword pooling** for subword → word alignment.

In [ ]:
!pip install transformers datasets seqeval accelerate pyyaml -q

In [ ]:
!git clone https://github.com/nam13092003/NER-financial-data.git /kaggle/working/NER_finance

In [ ]:
import yaml

CONFIG_PATH = "/kaggle/working/NER_finance/NER_Token_Classification/configs/default.yaml"
MY_CONFIG   = "/kaggle/working/my_config_token.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ── Change experiment settings here ──────────────────────────
# cfg["model"]["name"]       = "roberta-base"
# cfg["model"]["name"]       = "microsoft/deberta-v3-base"
cfg["model"]["name"]       = "bert-base-uncased"
cfg["model"]["max_seq_length"] = 256
# ─────────────────────────────────────────────────────────────

# FIRE dataset files
cfg["data"]["train_files"] = [
    {"path": "/kaggle/input/fire-dataset/fire_train.json"},
]
cfg["data"]["eval_files"] = [
    {"path": "/kaggle/input/fire-dataset/fire_dev.json"},
]
cfg["data"]["test_files"] = [
    {"path": "/kaggle/input/fire-dataset/fire_test.json"},
]

cfg["training"]["batch_size"] = 16
cfg["training"]["epochs"] = 10
cfg["training"]["learning_rate"] = 5e-5
cfg["training"]["eval_steps"] = 100
cfg["training"]["save_steps"] = 100
cfg["training"]["early_stopping_patience"] = 3

with open(MY_CONFIG, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("Config saved to:", MY_CONFIG)

In [ ]:
CONFIG = "/kaggle/working/my_config_token.yaml"

!python /kaggle/working/NER_finance/NER_Token_Classification/train.py \
    --config {CONFIG} \
    2>&1 | tee /kaggle/working/training_token_log.txt

In [ ]:
CONFIG    = "/kaggle/working/my_config_token.yaml"
CHECKPOINT = "/kaggle/working/outputs_ner_token/final_model"

!python /kaggle/working/NER_finance/NER_Token_Classification/evaluate.py \
    --config {CONFIG} \
    --checkpoint {CHECKPOINT} \
    --split test \
    --batch_size 32 \
    --output /kaggle/working/eval_results_token.json \
    2>&1 | tee /kaggle/working/eval_token_log.txt

In [ ]:
# ── (Optional) Evaluate on dev set ───────────────────────────
# CONFIG    = "/kaggle/working/my_config_token.yaml"
# CHECKPOINT = "/kaggle/working/outputs_ner_token/final_model"

# !python /kaggle/working/NER_finance/NER_Token_Classification/evaluate.py \
#     --config {CONFIG} \
#     --checkpoint {CHECKPOINT} \
#     --split eval \
#     --batch_size 32 \
#     --output /kaggle/working/eval_results_token_dev.json \
#     2>&1 | tee /kaggle/working/eval_token_dev_log.txt